In [1]:
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from functools import partial
import cv2
from tqdm import tqdm

In [2]:
from sample_frames_ecc import ecc_warp, choose_metric_size

In [3]:
frames = list(Path("/home/roman/ba/niederfuellbach2_autos/frames/mercedes_a_class_orbit_frames_1_sdr").glob("*.png"))

In [4]:
pairs = list(zip(frames,frames[1:]))

In [5]:
metric_wh = choose_metric_size(
    first_frame_bgr=cv2.imread(frames[0]),
    target_w=960,
    target_h=540,
)

In [6]:
ecc_fn = partial(
    ecc_warp,
    model="homography",
    metric_wh=metric_wh,
)

In [8]:
with ThreadPoolExecutor() as executor:
    ecc_dist = list(tqdm(executor.map(ecc_fn, pairs), total=len(pairs)))

100%|██████████| 14426/14426 [33:05<00:00,  7.27it/s]


In [11]:
ecc_dist

[0.6961519718170166,
 0.8087875843048096,
 0.7927371263504028,
 0.8430817127227783,
 0.8589888215065002,
 0.917831540107727,
 0.8878285884857178,
 0.8401846289634705,
 0.6817516684532166,
 0.5775970220565796,
 0.5164409875869751,
 0.5961577892303467,
 0.766461968421936,
 0.8353706002235413,
 1.0233314037322998,
 1.1405762434005737,
 1.234168291091919,
 1.2642285823822021,
 1.1831691265106201,
 1.1765117645263672,
 1.0758614540100098,
 0.887202262878418,
 0.7665752172470093,
 0.6686540842056274,
 0.7191393971443176,
 0.8490639925003052,
 0.9055521488189697,
 0.8515249490737915,
 0.821715772151947,
 0.7856708765029907,
 0.8401279449462891,
 0.767518162727356,
 0.720835268497467,
 0.7265685796737671,
 0.7312461137771606,
 0.7869759798049927,
 0.9630367159843445,
 1.1504783630371094,
 1.3670402765274048,
 1.493293285369873,
 1.615232229232788,
 1.7305326461791992,
 1.8436918258666992,
 2.162283182144165,
 2.4953079223632812,
 5.037310600280762,
 5.557880401611328,
 5.630001544952393,
 5.38

In [10]:
d = [d[0] for d in ecc_dist]
err = [d[1] for d in ecc_dist]

TypeError: 'float' object is not subscriptable

In [14]:
# df = pd.DataFrame({"dist": d + [0.0], "err": err + [1.0], "frame1": frames})
df = pd.DataFrame({"dist": ecc_dist + [0.0], "frame1": frames})

In [15]:
df.to_pickle("/home/roman/ba/mercedes_ecc_homography2.pkl")

In [52]:
pkls = list(Path("/home/roman/ba/niederfuellbach2_autos/frame_distances").iterdir())

In [63]:
pd.read_pickle(pkls[0])

,name,sharpness,dist
0,/home/roman/ba/niederfuellbach2_autos/frames/m...,1128.268878,1.378382
1,/home/roman/ba/niederfuellbach2_autos/frames/m...,1211.548382,1.344790
2,/home/roman/ba/niederfuellbach2_autos/frames/m...,1233.124155,1.235398
3,/home/roman/ba/niederfuellbach2_autos/frames/m...,1203.057104,1.520507
4,/home/roman/ba/niederfuellbach2_autos/frames/m...,1218.534970,1.992893
...,...,...,...
1735,/home/roman/ba/niederfuellbach2_autos/frames/m...,3975.626043,5.754722
1736,/home/roman/ba/niederfuellbach2_autos/frames/m...,4585.837999,7.408754
1737,/home/roman/ba/niederfuellbach2_autos/frames/m...,4454.864914,8.909838
1738,/home/roman/ba/niederfuellbach2_autos/frames/m...,4925.961714,9.152380


In [73]:
for p in pkls:
    df = pd.read_pickle(p)
    if isinstance(df["dist"].values[0], tuple):
        print("Processing ", p)
        df["dist"] = df["dist"].apply(lambda x: x[0] if isinstance(x, tuple) else x)
    else:
        print("dist is not a tuple ", p)
    df = df.reset_index(drop=True)
    df.to_pickle(p)

dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/mercedes_a_class_closeup_damage_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/mercedes_a_class_closeup_damage2_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/mercedes_a_class_closeup_damage3_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/mercedes_a_class_orbit_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/white_van_orbit_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/yellow_van_closeup_damage_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/yellow_van_closeup_damage2_frames_1_sdr.pkl
dist is not a tuple  /home/roman/ba/niederfuellbach2_autos/frame_distances/yellow_van_orbit_frames_1_sdr.pkl


In [74]:
dff = pd.read_pickle("/home/roman/ba/niederfuellbach2_autos/frame_distances/mercedes_a_class_closeup_damage2_frames_1_sdr.pkl")

In [78]:
dff["dist"]

0       2.379302
1       2.340890
2       2.267688
3       2.095264
4       1.880408
          ...   
1323    5.692407
1324    5.854483
1325    5.699860
1326    5.343443
1327    4.857228
Name: dist, Length: 1328, dtype: float64

In [80]:
pd.to_numeric(dff["dist"], errors="coerce")

0       2.379302
1       2.340890
2       2.267688
3       2.095264
4       1.880408
          ...   
1323    5.692407
1324    5.854483
1325    5.699860
1326    5.343443
1327    4.857228
Name: dist, Length: 1328, dtype: float64

In [50]:
dff

,name,sharpness,dist
0,/home/roman/ba/niederfuellbach2_autos/frames/m...,3281.816895,2.379302
1,/home/roman/ba/niederfuellbach2_autos/frames/m...,3198.941683,2.340890
2,/home/roman/ba/niederfuellbach2_autos/frames/m...,3306.335809,2.267688
3,/home/roman/ba/niederfuellbach2_autos/frames/m...,3219.024172,2.095264
4,/home/roman/ba/niederfuellbach2_autos/frames/m...,3349.150118,1.880408
...,...,...,...
1626,/home/roman/ba/niederfuellbach2_autos/frames/m...,1715.278304,5.692407
1627,/home/roman/ba/niederfuellbach2_autos/frames/m...,1661.700856,5.854483
1628,/home/roman/ba/niederfuellbach2_autos/frames/m...,1666.152493,5.699860
1629,/home/roman/ba/niederfuellbach2_autos/frames/m...,1383.083551,5.343443
